In [1]:
from datetime import datetime

from libs import (
    test_acceleration,
    print_estimation,
    bootstrap_growth_parameters,
    bootstrap_doubling_time_fixed,
    sliding_window_analysis,
)
from model_data import model_data

print("Loaded libraries")

Loaded libraries


In [2]:
# Construct observed_models from model_data
# Format: (model_name, release_date, task_length_50%, task_length_80%) in hours
observed_models = sorted([
    (model['name'], model['launch_date'], model['performance_50p'], model['performance_80p'])
    for model in model_data.values()
    if model['performance_50p'] is not None],
    key=lambda x: x[1])

print("=== 50% Reliability ===")
print_estimation(observed_models)
print_estimation(observed_models[1:])
print_estimation(observed_models[3:])
print_estimation(observed_models[5:])

print_estimation(observed_models[1:-1])
print_estimation(observed_models[3:-1])
print_estimation(observed_models[5:-1])
print_estimation(observed_models[5:-1])

print("\n=== 80% Reliability ===")
print_estimation(observed_models, "80%")
print_estimation(observed_models[1:], "80%")
print_estimation(observed_models[3:], "80%")
print_estimation(observed_models[5:], "80%")

print_estimation(observed_models[1:-1], "80%")
print_estimation(observed_models[3:-1], "80%")
print_estimation(observed_models[5:-1], "80%")
print_estimation(observed_models[5:-1], "80%")

=== 50% Reliability ===
GPT-2 to GPT 5.2 (50%): (336, 0.9)
GPT-3 (davinci-002) to GPT 5.2 (50%): (300, 0.9)
GPT-4 to GPT 5.2 (50%): (209, 0.9)
Claude 3 Opus to GPT 5.2 (50%): (97, 1.0)
GPT-3 (davinci-002) to Claude 4.5 Opus (50%): (300, 0.9)
GPT-4 to Claude 4.5 Opus (50%): (210, 0.9)
Claude 3 Opus to Claude 4.5 Opus (50%): (98, 1.0)
Claude 3 Opus to Claude 4.5 Opus (50%): (98, 1.0)

=== 80% Reliability ===
GPT-2 to GPT 5.2 (80%): (330, 0.901)
GPT-3 (davinci-002) to GPT 5.2 (80%): (311, 0.9)
GPT-4 to GPT 5.2 (80%): (211, 0.9)
Claude 3 Opus to GPT 5.2 (80%): (92, 1.0)
GPT-3 (davinci-002) to Claude 4.5 Opus (80%): (312, 0.9)
GPT-4 to Claude 4.5 Opus (80%): (212, 0.9)
Claude 3 Opus to Claude 4.5 Opus (80%): (92, 1.0)
Claude 3 Opus to Claude 4.5 Opus (80%): (92, 1.0)


In [3]:
print("=== Bootstrap Analysis ===")
current_date = datetime.today()
print(f"Current date: {current_date.strftime('%Y-%m-%d')}")

for rel in ["50%", "80%"]:
    print(f"\n{rel} Reliability:")
    full_results = bootstrap_growth_parameters(
        observed_models, reliability_level=rel, current_date=current_date
    )
    print(f"Full dataset: {full_results['median']} (95% CI: {full_results['ci_95']})")
    recent_models = [m for m in observed_models if m[1] >= datetime(2024, 1, 1)]
    recent_results = bootstrap_growth_parameters(
        recent_models, reliability_level=rel, current_date=current_date
    )
    print(
        f"2024+ models: {recent_results['median']} (95% CI: {recent_results['ci_95']})"
    )
    very_recent_models = [m for m in observed_models if m[1] >= datetime(2025, 1, 1)]
    very_recent_results = bootstrap_growth_parameters(
        very_recent_models, reliability_level=rel, current_date=current_date
    )
    print(
        f"2025+ models: {very_recent_results['median']} (95% CI: {very_recent_results['ci_95']})"
    )

    # No-acceleration variants (acceleration fixed at 1.0)
    full_fixed = bootstrap_doubling_time_fixed(
        observed_models, reliability_level=rel, current_date=current_date
    )
    print(f"Full dataset (no accel): {full_fixed['median']} (95% CI: {full_fixed['ci_95']})")
    recent_fixed = bootstrap_doubling_time_fixed(
        recent_models, reliability_level=rel, current_date=current_date
    )
    print(f"2024+ models (no accel): {recent_fixed['median']} (95% CI: {recent_fixed['ci_95']})")
    very_recent_fixed = bootstrap_doubling_time_fixed(
        very_recent_models, reliability_level=rel, current_date=current_date
    )
    print(f"2025+ models (no accel): {very_recent_fixed['median']} (95% CI: {very_recent_fixed['ci_95']})")

df = sliding_window_analysis(observed_models)
print("\n=== Parameter stability by time window ===")
print(df.groupby("window")[["doubling_time", "acceleration"]].agg(["mean", "std"]))

=== Bootstrap Analysis ===
Current date: 2026-02-07

50% Reliability:
Full dataset: (318, 0.9) (95% CI: {'doubling_time': (145, 340), 'acceleration': (0.9, 0.923)})
2024+ models: (99, 1.0) (95% CI: {'doubling_time': (86, 130), 'acceleration': (0.9, 1.0)})
2025+ models: (120, 1.0) (95% CI: {'doubling_time': (69, 168), 'acceleration': (0.9, 1.0)})
Full dataset (no accel): 203 (95% CI: (121, 215))
2024+ models (no accel): 98 (95% CI: (85, 113))
2025+ models (no accel): 116 (95% CI: (64, 165))

80% Reliability:
Full dataset: (310, 0.9) (95% CI: {'doubling_time': (148, 336), 'acceleration': (0.9, 0.948)})
2024+ models: (94, 1.0) (95% CI: {'doubling_time': (88, 118), 'acceleration': (0.919, 1.0)})
2025+ models: (128, 1.0) (95% CI: {'doubling_time': (72, 207), 'acceleration': (0.9, 1.0)})
Full dataset (no accel): 204 (95% CI: (121, 216))
2024+ models (no accel): 93 (95% CI: (87, 114))
2025+ models (no accel): 123 (95% CI: (70, 203))

=== Parameter stability by time window ===
       doubling_

In [4]:
test_acceleration(
    start_task_length=model_data['gpt2']['performance_50p'],
    agi_task_length=167,
    initial_doubling_time=260,
    acceleration=0.95,
    start_date=model_data['gpt2']['launch_date'].strftime('%Y-%m-%d'),
)

Step |    Date    |    Day |       Task | τ (d)
-----------------------------------------------
   0 | 2019‑02‑14 |      0 |       2sec | 260.0
   1 | 2019‑11‑01 |    260 |       5sec | 247.0
   2 | 2020‑07‑05 |    507 |      10sec | 234.6
   3 | 2021‑02‑24 |    741 |      19sec | 222.9
   4 | 2021‑10‑05 |    964 |      38sec | 211.8
   5 | 2022‑05‑05 |   1176 |    1.27min | 201.2
   6 | 2022‑11‑22 |   1377 |    2.54min | 191.1
   7 | 2023‑06‑01 |   1568 |    5.09min | 181.6
   8 | 2023‑11‑30 |   1750 |   10.18min | 172.5
   9 | 2024‑05‑20 |   1922 |   20.36min | 163.9
  10 | 2024‑10‑31 |   2086 |   40.72min | 155.7
  11 | 2025‑04‑05 |   2242 |     1.36hr | 147.9
  12 | 2025‑08‑31 |   2390 |     2.71hr | 140.5
  13 | 2026‑01‑18 |   2530 |     5.43hr | 133.5
  14 | 2026‑06‑01 |   2664 |    10.86hr | 126.8
  15 | 2026‑10‑05 |   2790 |    21.72hr | 120.5
  16 | 2027‑02‑03 |   2911 |    43.43hr | 114.4
  17 | 2027‑05‑28 |   3025 |    86.86hr | 108.7
  18 | 2027‑09‑14 |   3134 |   173.72hr 

In [5]:
test_acceleration(
    start_task_length=model_data['gpt2']['performance_50p'],
    agi_task_length=167,
    initial_doubling_time=260,
    acceleration=0.95,
    start_date=model_data['gpt2']['launch_date'].strftime('%Y-%m-%d'),
)

Step |    Date    |    Day |       Task | τ (d)
-----------------------------------------------
   0 | 2019‑02‑14 |      0 |       2sec | 260.0
   1 | 2019‑11‑01 |    260 |       5sec | 247.0
   2 | 2020‑07‑05 |    507 |      10sec | 234.6
   3 | 2021‑02‑24 |    741 |      19sec | 222.9
   4 | 2021‑10‑05 |    964 |      38sec | 211.8
   5 | 2022‑05‑05 |   1176 |    1.27min | 201.2
   6 | 2022‑11‑22 |   1377 |    2.54min | 191.1
   7 | 2023‑06‑01 |   1568 |    5.09min | 181.6
   8 | 2023‑11‑30 |   1750 |   10.18min | 172.5
   9 | 2024‑05‑20 |   1922 |   20.36min | 163.9
  10 | 2024‑10‑31 |   2086 |   40.72min | 155.7
  11 | 2025‑04‑05 |   2242 |     1.36hr | 147.9
  12 | 2025‑08‑31 |   2390 |     2.71hr | 140.5
  13 | 2026‑01‑18 |   2530 |     5.43hr | 133.5
  14 | 2026‑06‑01 |   2664 |    10.86hr | 126.8
  15 | 2026‑10‑05 |   2790 |    21.72hr | 120.5
  16 | 2027‑02‑03 |   2911 |    43.43hr | 114.4
  17 | 2027‑05‑28 |   3025 |    86.86hr | 108.7
  18 | 2027‑09‑14 |   3134 |   173.72hr 